In [2]:
color_map = {
    'Fibroblasts': '#9467bd',
 'Tumor': '#d62728',
 'High_Endothelial_Venules': '#ff7f0e',
 'M1_macrophages': '#ffbb78',
 'CAF': '#2ca02c',
 'Plasma_IgG': '#98df8a',
 'unknown': '#aec7e8',
 'CAM': '#ff9896',
 'Germinal_Center_Plasma_IgM_B_cell': '#1f77b4',
 'Plasma_IgA': '#c5b0d5',
 'T_cell': '#8c564b',
 'Tumor_Keratin_Pearl': '#c49c94',
 'Macrophages': '#e377c2',
 'Cytotoxic_IFN_signaling': '#f7b6d2',
 'Cortex_CCL21': '#7f7f7f'
}

# scGPT fine tuning

In [3]:
import scanpy as sc 
adata = sc.read_h5ad('/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/scGPT/HumanLymph/adata_test_scGPT.h5ad')
adata 

AnnData object with n_obs × n_vars = 39167 × 21040
    obs: 'cell_ID_mask', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_reads', 'reads_per_counts', 'n_joined', 'exact_entropy', 'theoretical_entropy', 'exact_compression', 'theoretical_compression', 'n_counts', 'annotation', 'annotation_key', 'n_section', 'original_clusters', 'Z', 'batch', 'batch_id', 'str_batch', 'celltype', 'celltype_id'
    var: 'gene_name', 'id_in_vocab'
    obsm: 'X_scGPT', 'bin_edges', 'spatial', 'spatial_3d_aligned'
    layers: 'X_binned', 'X_normed', 'counts', 'raw'

In [4]:
import pickle

path = "/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/scGPT/HumanLymph/results.pkl"

with open(path, "rb") as f:
    results = pickle.load(f)

import numpy as np
import pandas as pd

# 从 results 中取出
y_pred = results["predictions"]      # numpy array of ints
id_maps = results["id_maps"]          # dict: int -> str

# 映射成字符串
y_pred_str = np.array([id_maps[int(i)] for i in y_pred])

# 写入 adata.obs
adata.obs["predictions"] = y_pred_str

# 设为 category（强烈推荐，方便配色 & 下游分析）
adata.obs["predictions"] = adata.obs["predictions"].astype("category")
# 按 id_maps 的 key 顺序定义 categories
categories = [id_maps[i] for i in sorted(id_maps.keys())]

adata.obs["predictions"] = pd.Categorical(
    adata.obs["predictions"],
    categories=categories
)

In [5]:
np.sum(adata.obs["predictions"] == adata.obs["original_clusters"])

26328

In [4]:
import clusim.sim as sim
from clusim.clustering import Clustering
import numpy as np
# conda activate SPACEL

import pandas as pd
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics import confusion_matrix
from scipy.spatial.distance import *
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
import scanpy as sc
import os, os.path as osp
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn import metrics
def cons_coef(adata1, adata2, cluster1, cluster2, return_all=True, alpha=.9):
    obs_names = adata1.obs_names
    c1 = Clustering(elm2clu_dict = {index: [value] for index, value in zip(obs_names, adata1.obs[cluster1])})
    
    c2 = Clustering(elm2clu_dict = {index: [value] for index, value in zip(obs_names, adata2.obs[cluster2])})
    sis, relabeled_elements = sim.element_sim_elscore(c1, c2, alpha=alpha)
    
    sis_mapped = np.zeros(len(obs_names))

    for obs_index, score_position in relabeled_elements.items():
        sis_mapped[obs_names.get_loc(obs_index)] = sis[score_position]
        
    if return_all:
        return sis_mapped, np.mean(sis_mapped)
    else:
        return sis_mapped

In [ ]:
df = pd.read_csv('/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/scGPT/HumanLymph_reference_mapping/predictions_results.csv', index_col=0)
adata.obs_names = df.index
st_data = adata 
def ASW_score(X, pred_labels):
    d = squareform(pdist(X))
    return silhouette_score(X=d,labels=pred_labels,metric='precomputed')
st_data = st_data[df.index, :].copy()
out_rows = []
X = st_data.obsm['spatial']
y_true = st_data.obs['original_clusters'].astype(str).to_numpy()
#y_pred = df['predictions'].astype(str).to_numpy()

st_data.obs['Predicted'] = st_data.obs['predictions'].astype(str).to_numpy()
#y_true = np.char.replace(y_true, '_', '/')
#y_pred = np.char.replace(y_pred, '_', '/')
labels = np.unique(y_true)  # ensure we average over true classes
precision_macro = precision_score(
    y_true, y_pred, labels=labels, average="macro", zero_division=0
)

labels = np.unique(np.concatenate([y_true, y_pred]))
le = LabelEncoder().fit(labels)
yt = le.transform(y_true)
yp = le.transform(y_pred)
mask = (yt == yp)
n_correct = np.count_nonzero(mask)   
n_total = mask.size
acc = n_correct / n_total
X = st_data.obsm['spatial']
asw = ASW_score(X=X, pred_labels= y_pred)
print(f"correct: {n_correct}/{n_total} ({acc:.2%})")
asw = (asw + 1.0) / 2.0
asw = max(0.0, min(1.0, float(asw)))
# -------- series 级 F1（保持你原逻辑）--------
f1_macro    = f1_score(yt, yp, average='macro')
f1_weighted = f1_score(yt, yp, average='weighted')
f1_micro    = f1_score(yt, yp, average='micro')  
sis_map, mean_val = cons_coef(st_data, st_data, 'original_clusters', 'Predicted', return_all=True, alpha=.9)
ari = adjusted_rand_score(yp, yt)
recall_macro = recall_score(y_true, y_pred, average="macro") 
ami = metrics.adjusted_mutual_info_score(y_true, y_pred)
nmi = metrics.normalized_mutual_info_score(y_true, y_pred)
avgbio = (nmi + ari + asw) / 3.0 if np.isfinite(asw) else np.nan

out_rows.append({
    'Method': 'scGPT', 
    'accuracy': acc,
    'f1_macro': f1_macro,
    'f1_weighted': f1_weighted,
    'f1_micro': f1_micro,
    'precision': precision_macro,
    'recall': recall_macro,
    'ARI': ari,
    'AMI': ami,
    'NMI': nmi,
    'ASW': asw,
    'AvgBIO': avgbio,
    'ECS': mean_val
})
results_df = pd.DataFrame(out_rows)
output_path = '/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/scGPT/HumanLymph/metrics_results.csv'
results_df.to_csv(output_path, index=False)

In [11]:
adata.write_h5ad('/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/scGPT/HumanLymph/processed.h5ad')

In [6]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# -----------------------------
# Sanity checks
# -----------------------------
assert "X_scGPT" in adata.obsm
assert "original_clusters" in adata.obs
assert "predictions" in adata.obs

# -----------------------------
# Build neighbors + UMAP once
# -----------------------------
sc.pp.neighbors(
    adata,
    use_rep="X_scGPT",
    n_neighbors=15,
    metric="cosine"
)

sc.tl.umap(adata, min_dist=0.3, random_state=42)

# -----------------------------
# Ensure categorical + ordering
# -----------------------------
adata.obs["original_clusters"] = adata.obs["original_clusters"].astype("category")
adata.obs["predictions"] = adata.obs["predictions"].astype("category")

# 强制 category 顺序与 color_map 一致（论文级稳定性）
ordered_categories = list(color_map.keys())

adata.obs["original_clusters"] = adata.obs["original_clusters"].cat.set_categories(
    ordered_categories
)
adata.obs["predictions"] = adata.obs["predictions"].cat.set_categories(
    ordered_categories
)

# -----------------------------
# Create side-by-side UMAP
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Annotated (ground truth)
sc.pl.umap(
    adata,
    color="original_clusters",
    palette=color_map,
    size=8,
    alpha=0.7,
    frameon=False,
    legend_loc= None,
    title="Annotated",
    ax=axes[0],
    show=False
)

# Right: Predicted
sc.pl.umap(
    adata,
    color="predictions",
    palette=color_map,
    size=8,
    alpha=0.7,
    frameon=False,
    legend_loc="right margin",
    title="Predicted",
    ax=axes[1],
    show=False
)

# -----------------------------
# Save as PDF
# -----------------------------
plt.tight_layout()
plt.savefig(
    "/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/scGPT/HumanLymph/scGPT_umap_annotated_vs_predicted0130.pdf",
    format="pdf",
    bbox_inches="tight"
)
plt.close()


/home/zhuy45/miniconda3/envs/SPACEL/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zhuy45/miniconda3/envs/SPACEL/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/zhuy45/miniconda3/envs/SPACEL/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
